In [0]:
import re
from pyspark.sql import functions as F

SOURCE = "abfss://alsource@footballmanagerli.dfs.core.windows.net/"
DEST   = "abfss://aldestination@footballmanagerli.dfs.core.windows.net"

def to_snake(c):
    c = re.sub(r"[^0-9a-zA-Z]+", "_", c)
    c = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", c)
    return re.sub(r"_+", "_", c).strip("_").lower()

def start_stream(filename):
    table = to_snake(filename.rsplit(".", 1)[0])        # "game_team_stats.csv" -> "game_team_stats"

    raw = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{DEST}/schemas/{table}")
        .option("pathGlobFilter", filename)              # only this file
        .load(SOURCE))

    df = (raw.toDF(*[to_snake(c) for c in raw.columns])
             .withColumn("_source_file", F.col("_metadata.file_path"))
             .withColumn("_ingested_at", F.current_timestamp()))

    return (df.writeStream
        .format("delta")
        .queryName(table)                                # shows up by name in the Spark UI
        .option("checkpointLocation", f"{DEST}/checkpoint/{table}")
        .trigger(availableNow=True)
        .start(f"{DEST}/data/{table}"))

# Find every CSV in alsource and start a stream for each
csv_files = [f.name for f in dbutils.fs.ls(SOURCE) if f.name.lower().endswith(".csv")]
print("Streaming:", csv_files)

queries = {name: start_stream(name) for name in csv_files}

In [0]:
from delta.tables import DeltaTable

CATALOG = "football_manager"
SCHEMA  = "nfl"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

for filename, q in queries.items():
    table     = q.name
    path      = f"{DEST}/data/{table}"
    full_name = f"{CATALOG}.{SCHEMA}.{table}"      # e.g. football_manager.nfl.game_team_stats

    try:
        q.processAllAvailable()
    except Exception as e:
        print(f"{filename}: stream failed → {e}")
        continue

    if not DeltaTable.isDeltaTable(spark, path):
        print(f"{filename}: no data written yet, skipping")
        continue

    spark.sql(f"CREATE TABLE IF NOT EXISTS {full_name} LOCATION '{path}'")

display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))